In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import warnings, os, pickle
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    logging as hf_logging,
)
from collections import Counter
from datetime import datetime

hf_logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

EMO_MODEL_NAME = "vinai/phobert-base"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

Using device: cuda


In [3]:
EMOTION_GROUPS: dict[str, dict] = {
    # ===== POSITIVE =====
    "joy":            {"category": "positive",  "score":  1.0, "guidance": "vui mừng, hạnh phúc", "icon": ("😊", "#f59e0b")},
    "amusement":      {"category": "positive",  "score":  1.0, "guidance": "vui vẻ, hài hước", "icon": ("😄", "#fbbf24")},
    "excitement":     {"category": "positive",  "score":  1.0, "guidance": "hứng khởi, phấn khích", "icon": ("🤩", "#f97316")},
    "optimism":       {"category": "positive",  "score":  1.0, "guidance": "lạc quan, tin tưởng", "icon": ("🌟", "#84cc16")},
    "pride":          {"category": "positive",  "score":  1.0, "guidance": "tự hào, tự tôn", "icon": ("💪", "#10b981")},
    "admiration":     {"category": "positive",  "score":  1.0, "guidance": "kính trọng, ngưỡng mộ", "icon": ("🌸", "#a78bfa")},
    "gratitude":      {"category": "positive",  "score":  1.0, "guidance": "biết ơn, trân trọng", "icon": ("🙏", "#34d399")},
    "relief":         {"category": "positive",  "score":  1.0, "guidance": "thoải mái, an tâm", "icon": ("😌", "#6ee7b7")},
    "approval":       {"category": "positive",  "score":  1.0, "guidance": "đồng ý, ủng hộ", "icon": ("👍", "#4ade80")},
    # ===== AFFECTION =====
    "love":           {"category": "affection", "score":  1.5, "guidance": "yêu thương, tình cảm", "icon": ("❤️", "#f43f5e")},
    "desire":         {"category": "affection", "score":  1.5, "guidance": "khao khát, mong muốn", "icon": ("💫", "#ec4899")},
    "caring":         {"category": "affection", "score":  1.5, "guidance": "quan tâm, chăm sóc", "icon": ("🤗", "#fb7185")},
    # ===== NEUTRAL =====
    "realization":    {"category": "neutral",   "score":  0.0, "guidance": "nhận thức, hiểu biết", "icon": ("💡", "#60a5fa")},
    "surprise":       {"category": "neutral",   "score":  0.0, "guidance": "bất ngờ, ngạc nhiên", "icon": ("😮", "#818cf8")},
    "curiosity":      {"category": "neutral",   "score":  0.0, "guidance": "tò mò, khám phá", "icon": ("🔍", "#38bdf8")},
    "neutral":        {"category": "neutral",   "score":  0.0, "guidance": "trung lập, bình thản", "icon": ("😐", "#94a3b8")},
    # ===== CONFUSION =====
    "confusion":      {"category": "confusion", "score": -0.5, "guidance": "bối rối, nhầm lẫn", "icon": ("🤔", "#fcd34d")},
    # ===== ANXIETY =====
    "fear":           {"category": "anxiety",   "score": -1.0, "guidance": "sợ hãi, lo lắng", "icon": ("😰", "#7dd3fc")},
    "nervousness":    {"category": "anxiety",   "score": -1.0, "guidance": "bồn chồn, căng thẳng", "icon": ("😟", "#93c5fd")},
    # ===== SADNESS =====
    "remorse":        {"category": "sadness",   "score": -1.0, "guidance": "hối hận, ăn năn", "icon": ("😔", "#a5b4fc")},
    "embarrassment":  {"category": "sadness",   "score": -1.0, "guidance": "xấu hổ, tự ti", "icon": ("😳", "#fda4af")},
    "disappointment": {"category": "sadness",   "score": -1.0, "guidance": "thất vọng, chán nản", "icon": ("😞", "#6b7280")},
    "sadness":        {"category": "sadness",   "score": -1.0, "guidance": "buồn bã, trầm lắng", "icon": ("😢", "#64748b")},
    "grief":          {"category": "sadness",   "score": -1.0, "guidance": "đau buồn, mất mát", "icon": ("💔", "#475569")},
    # ===== ANGER =====
    "disgust":        {"category": "anger",     "score": -1.5, "guidance": "ghê tởm, khó chịu", "icon": ("🤢", "#4b5563")},
    "anger":          {"category": "anger",     "score": -1.5, "guidance": "tức giận, bực bội", "icon": ("😠", "#ef4444")},
    "annoyance":      {"category": "anger",     "score": -1.5, "guidance": "phiền phức, khó chịu", "icon": ("😤", "#f87171")},
    "disapproval":    {"category": "anger",     "score": -1.5, "guidance": "bất đồng, phản đối", "icon": ("👎", "#dc2626")},
}

# labels.py

# =========================================================
# EMOTION LABELS
# =========================================================

EMOTIONS = [
    "amusement",
    "excitement",
    "joy",
    "love",
    "desire",
    "optimism",
    "caring",
    "pride",
    "admiration",
    "gratitude",
    "relief",
    "approval",
    "realization",
    "surprise",
    "curiosity",
    "confusion",
    "fear",
    "nervousness",
    "remorse",
    "embarrassment",
    "disappointment",
    "sadness",
    "grief",
    "disgust",
    "anger",
    "annoyance",
    "disapproval",
    "neutral"
]

# =========================================================
# EMOTION <-> ID
# =========================================================

emotion2id = {
    emotion: idx
    for idx, emotion in enumerate(EMOTIONS)
}

id2emotion = {
    idx: emotion
    for idx, emotion in enumerate(EMOTIONS)
}

NUM_EMOTIONS = len(EMOTIONS)

# =========================================================
# EMOTION GROUPS
# =========================================================

# The EMOTION_GROUPS dictionary is defined twice in this cell. The first definition
# is for detailed information about each emotion, including category, score, guidance,
# and icon. The second definition, which is currently present, maps each emotion
# directly to its broader category. While both serve to group emotions, the first
# one is more comprehensive and seems to be the intended primary source of emotion
# data for the chatbot's logic, particularly for scoring and guidance. To avoid
# redundancy and potential confusion, and to ensure the chatbot uses the richer
# emotion data, I will remove the second, simpler definition of EMOTION_GROUPS
# and rely solely on the first one, which is already correctly typed as
# `dict[str, dict]` and includes all necessary information.

# =========================================================
# BEHAVIOR LABELS
# =========================================================

BEHAVIORS = sorted(
    list(set(emo_info["category"] for emo_info in EMOTION_GROUPS.values()))
)

behavior2id = {
    behavior: idx
    for idx, behavior in enumerate(BEHAVIORS)
}

id2behavior = {
    idx: behavior
    for idx, behavior in enumerate(BEHAVIORS)
}

NUM_BEHAVIORS = len(BEHAVIORS)

In [4]:

class EmotionModel(nn.Module):

    def __init__(self, n_emo, n_behavior=None, base_model=EMO_MODEL_NAME):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model)
        h = self.encoder.config.hidden_size
        self.emotion_head = self._head(h, n_emo)
        self.behavior_head = (
            self._head(h, n_behavior)
            if n_behavior is not None else None
        )

    @staticmethod
    def _head(hidden, n_out):
        return nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, n_out),
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = out.last_hidden_state[:, 0]
        logits = {
            "emotion": self.emotion_head(cls)
        }
        if self.behavior_head is not None:
            logits["behavior"] = self.behavior_head(cls)
        return logits

    @torch.no_grad()
    def predict(
        self,
        text: str,
        tokenizer,
        id2emotion,
        id2behavior=None,
        threshold=0.5
    ):

        self.eval()

        device = next(self.parameters()).device

        batch = tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt",
        )

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        # =====================================
        # MULTI LABEL EMOTION
        # =====================================

        probs = torch.sigmoid(
            logits["emotion"]
        )[0]

        pred_ids = (
            probs > threshold
        ).nonzero(as_tuple=True)[0].tolist()

        emotions = [
            id2emotion[i]
            for i in pred_ids
        ]

        result = {
            "emotion": emotions
        }

        # =====================================
        # BEHAVIOR
        # =====================================

        if (
            "behavior" in logits
            and id2behavior is not None
        ):

            behavior_id = logits[
                "behavior"
            ].argmax(-1).item()

            result["behavior"] = (
                id2behavior[behavior_id]
            )

        return result



In [5]:
from datasets import load_dataset

ds = load_dataset("Adapting/empathetic_dialogues_with_special_tokens")
docs = []

for row in ds["train"]:

    emotion = row["emotion"]

    situation = row["situation"]

    response = row["sys_response"]

    behavior = row["behavior"]

    text = f"""
Emotion: {emotion}

Behavior: {behavior}

Situation:
{situation}

Helpful Response:
{response}
""".strip()

    docs.append(text)

print("Total docs:", len(docs))

print("\nExample:\n")
print(docs[0])

Total docs: 40245

Example:

Emotion: faithful

Behavior: I'm in a positive mood, please congratulate me and praise me.

Situation:
I have always been a big fan of childrens place, I will never shop anywhere else

Helpful Response:
sounds nice. i am going to check that out myself


In [63]:
class Retriever:
    def __init__(self, model_name="intfloat/multilingual-e5-small"):
        print(f"Loading Embedding Model: {model_name}...")
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        self.matrix = None
        self.docs: list[str] = []

    @torch.no_grad()
    def _encode(self, texts: list[str]) -> torch.Tensor:

        encoded_input = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        ).to(device)

        model_output = self.model(**encoded_input)

        token_embeddings = (
            model_output.last_hidden_state
        )

        input_mask_expanded = (
            encoded_input["attention_mask"]
            .unsqueeze(-1)
            .expand(token_embeddings.size())
            .float()
        )

        embeddings = torch.sum(
            token_embeddings *
            input_mask_expanded,
            dim=1
        )

        embeddings = embeddings / torch.clamp(
            input_mask_expanded.sum(dim=1),
            min=1e-9
        )

        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

    def ingest(self, texts: list[str]) -> None:
        if not texts:
            print("Danh sách văn bản trống, không có gì để ingest.")
            return
        self.docs = texts
        all_embeddings = []
        batch_size = 64
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            all_embeddings.append(self._encode(batch_texts))

        self.matrix = torch.cat(all_embeddings, dim=0)
        print(f"Ingested {len(texts)} docs with {self.model_name}")

    @torch.no_grad()
    def search(self, query: str, k: int = 5) -> list[str]:
        if not query.strip() or self.matrix is None:
            return []

        q_vec = self._encode([query]) # (1, dim)
        scores = torch.mm(q_vec, self.matrix.t())[0] # (num_docs,)

        top_k = torch.topk(scores, k=min(k, len(self.docs)))
        indices = top_k.indices.cpu().tolist()
        values = top_k.values.cpu().tolist()

        return [
    {
        "text": self.docs[i],
        "score": float(score)
    }
    for i, score in zip(indices, values)
    if score > 0.15
]

    def save(self, path: str) -> None:
        os.makedirs(path, exist_ok=True)
        data = {
            "model_name": self.model_name,
            "matrix": self.matrix.cpu() if self.matrix is not None else None,
            "docs": self.docs
        }
        save_file = os.path.join(path, "retriever.pkl")
        with open(save_file, "wb") as f:
            pickle.dump(data, f)
        print(f"Retriever saved to {save_file}")

    @classmethod
    def load(cls, path: str) -> "Retriever":
        load_file = os.path.join(path, "retriever.pkl")
        with open(load_file, "rb") as f:
            d = pickle.load(f)

        r = cls(model_name=d["model_name"])
        r.docs = d["docs"]
        if d["matrix"] is not None:
            r.matrix = d["matrix"].to(device)
        print(f"Loaded BGE retriever: {len(r.docs)} docs")
        return r

retriever = Retriever()
retriever.ingest(docs)
retriever.save("/content/drive/MyDrive/Colab Notebooks/chat")

Loading Embedding Model: intfloat/multilingual-e5-small...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ingested 40245 docs with intfloat/multilingual-e5-small
Retriever saved to /content/drive/MyDrive/Colab Notebooks/chat/retriever.pkl


In [7]:
results = retriever.search(
    "tôi cảm thấy áp lực học tập",
    k=3
)

for r in results:
    print(r["score"])
    print(r["text"])
    print("="*50)

0.8339691162109375
Emotion: disappointed

Behavior: please give me some advices.

Situation:
I really needed a 34 on the ACT to get a big scholarship and I tried so many times. In the end, I got a 33 many times.

Helpful Response:
Oh, that is not good. You should have done better!
0.8316605687141418
Emotion: furious

Behavior: [None]

Situation:
I'm furious, all the classes I needed to enroll in this semester are filled up!

Helpful Response:
If you can't, it may throw you off by a whole semester!  I would be really mad if I were you!
0.8312207460403442
Emotion: afraid

Behavior: [None]

Situation:
Wasn't able to register for one of the classes I need to be on track to graduate in my last year of college. Have to contact the school and see if they'll be able to make an exception and squeeze me into a full class, or I don't know what I'll do.

Helpful Response:
Still, school is about making mistakes and learning from them


In [48]:
import os # Ensure os is imported for path operations

class LLM:
    def __init__(self, model_name: str = None, tokenizer=None, model=None):
        if tokenizer is not None and model is not None:
            # Initialize from provided tokenizer and model (for from_saved)
            self.tokenizer = tokenizer
            self.model = model
            self.model_name = model_name if model_name else "loaded_from_path"
            print("LLM instantiated from provided components.")
        elif model_name:
            # Standard initialization from model_name (e.g., Hugging Face ID)
            print(f"Loading LLM: {model_name} ...")
            dtype = torch.float16 if torch.cuda.is_available() else torch.float32
            self.model_name = model_name
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=dtype,
                device_map="auto",
                low_cpu_mem_usage=True,
            )
            self.model.eval()
            print("LLM loaded.")
        else:
            raise ValueError("Must provide either model_name or pre-loaded tokenizer and model.")

    @torch.no_grad()
    def generate(self, prompt: str, max_new_tokens: int = 150) -> str:
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        output = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.92,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        generated = output[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()

    def save(self, path: str) -> None:
        llm_dir = os.path.join(path, "llm")
        os.makedirs(llm_dir, exist_ok=True)
        self.tokenizer.save_pretrained(llm_dir)
        self.model.save_pretrained(llm_dir)
        # Save the original model_name for from_saved to know what to fall back to
        with open(os.path.join(llm_dir, "model_name.txt"), "w") as f:
            f.write(self.model_name)
        print(f"LLM saved to {llm_dir}")

    @classmethod
    def from_saved(cls, path: str, default_model_name: str = LLM_MODEL_NAME) -> "LLM":
        llm_dir = os.path.join(path, "llm")
        model_name_file = os.path.join(llm_dir, "model_name.txt")

        # Try to read the original model_name if available
        original_model_name = default_model_name
        if os.path.exists(model_name_file):
            with open(model_name_file, "r") as f:
                original_model_name = f.read().strip()

        # Check if the saved components actually exist
        if not os.path.exists(llm_dir) or not os.path.exists(os.path.join(llm_dir, "tokenizer.json")):
            print(f"WARNING: Saved LLM components not found at {llm_dir}. Initializing with default model: {original_model_name}.")
            return cls(model_name=original_model_name)

        print(f"Loading LLM from saved path: {llm_dir} ...")
        tokenizer = AutoTokenizer.from_pretrained(llm_dir)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            llm_dir,
            torch_dtype=dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        model.eval()
        print("LLM loaded from saved path.")
        return cls(model_name=original_model_name, tokenizer=tokenizer, model=model)

# lm = LLM(model_name=LLM_MODEL_NAME)
# lm.save("/content/drive/MyDrive/Colab Notebooks/chat")

In [56]:
class PromptBuilder:
    CRISIS_KEYWORDS = [
        "tự tử", "muốn chết", "không muốn sống", "kill myself", "suicide",
        "kết thúc cuộc đời", "biến mất mãi mãi", "chấm dứt cuộc sống",
        "đau khổ quá", "không chịu nổi nữa", "hết cách rồi",
    ]

    _DEFAULT_GUIDANCE = "chân thành, lắng nghe"

    def _is_crisis(self, text: str) -> bool:
        t = text.lower()
        return any(k in t for k in self.CRISIS_KEYWORDS)

    def _guidance(self, emotions: list[str]) -> str:
        for emo in emotions:
            # Đảm bảo EMOTION_GROUPS đã được định nghĩa ở scope bên ngoài
            info = EMOTION_GROUPS.get(emo.lower())
            if info:
                return info.get("guidance", self._DEFAULT_GUIDANCE)
        return self._DEFAULT_GUIDANCE

    def build(
        self,
        user_text: str,
        emotions: list[str],
        docs: list[str],
        history: list[dict] | None = None,
    ) -> str:
        guidance = self._guidance(emotions)

        # 1. Xử lý Context từ RAG (Tránh lỗi AttributeError: 'dict' object)
        context_lines = []
        if docs:
            for d in docs[:2]:
                # Trích xuất text an toàn từ dictionary hoặc string
                text = d.get('Situation', d.get('text', str(d))) if isinstance(d, dict) else d
                if text and text.strip():
                    context_lines.append(f"- {text[:120]}...")
        context = "\n".join(context_lines)

        # 2. Xử lý Lịch sử hội thoại
        hist_lines: list[str] = []
        if history:
            for h in history[-6:]:
                role = "Bạn" if h["role"] == "user" else "Mình"
                hist_lines.append(f"{role}: {h['content']}")
        hist_text = "\n".join(hist_lines)

        # 3. Kiểm tra khủng hoảng
        crisis_note = ""
        if self._is_crisis(user_text):
            crisis_note = (
                "\n[CẢNH BÁO NGUY CẤP] Người dùng có dấu hiệu khủng hoảng. "
                "Hãy phản hồi cực kỳ nhẹ nhàng, ưu tiên an toàn và nhắc về 1800 599 920.\n"
            )

        # 4. Xây dựng Prompt tổng thể
        parts = [
            "Bạn là một người bạn tâm giao đang lắng nghe người bạn thân nhất của mình chia sẻ.",
            "",
            "=== LUẬT VỀ THÁI ĐỘ (BẮT BUỘC) ===",
            "1. TUYỆT ĐỐI KHÔNG xem nhẹ nỗi buồn (không nói 'chỉ là...', 'có gì đâu mà...').",
            "2. KHÔNG tự suy diễn lý do ngoại cảnh (như thời tiết, ngày mưa) nếu người dùng không nhắc tới.",
            "3. Xưng 'Mình', gọi 'Bạn'. Ngôn ngữ tiếng Việt đời thường, ấm áp, KHÔNG máy móc.",
            "4. Nếu bạn mình buồn, hãy công nhận cảm xúc đó trước khi đặt câu hỏi gợi mở.",
            "5. CHỈ TRẢ LỜI BẰNG TIẾNG VIỆT – Tuyệt đối không dùng tiếng Trung hay bất kỳ ngôn ngữ nào khác.",
        ]

        # Đưa trạng thái cảm xúc vào logic
        emo_str = ", ".join(emotions) if emotions else "trung tính"
        parts.append(f"\n[Ngữ cảnh]: Bạn nhận thấy bạn mình đang cảm thấy: {emo_str}.")
        parts.append(f"[Hướng dẫn]: Phản hồi với sự {guidance}.")

        if crisis_note:
            parts.append(crisis_note)

        if context:
            parts.append(f"\n[Gợi ý đồng cảm - Dùng làm ý tưởng, không sao chép]:\n{context}")

        if hist_text:
            parts.append(f"\n[Dòng tâm sự cũ]:\n{hist_text}")

        # Kết thúc để LLM bắt đầu trả lời
        parts.append("\n---")
        parts.append(f"Bạn (người dùng): {user_text}")
        parts.append("Mình (người bạn tâm giao):")

        return "\n".join(parts)

In [57]:
class TherapyChatbot:
    def __init__(
        self,
        retriever: Retriever,
        emotion_model: EmotionModel,
        emo_tokenizer,
        id2emotion: dict,
        id2behavior: dict | None = None,
        llm: LLM | None = None
    ):

        self.retriever      = retriever
        self.emotion_model  = emotion_model.to(device).eval()
        self.emo_tokenizer  = emo_tokenizer
        self.id2emotion     = id2emotion
        self.id2behavior    = id2behavior
        self.llm            = llm if llm is not None else LLM(model_name=LLM_MODEL_NAME)
        self.prompt_builder = PromptBuilder()
        self.chat_history:    list[dict] = []
        self._emotion_history: list[list[str]] = []

    def chat(self, user_input: str, verbose: bool = False) -> str:
        analysis = self.emotion_model.predict(user_input, self.emo_tokenizer, self.id2emotion, self.id2behavior)
        emotions: list[str] = analysis["emotion"]
        self._emotion_history.append(emotions)
        docs_raw = self.retriever.search(user_input, k=5)

        # Trích xuất text từ dict để PromptBuilder không bị lỗi .strip()
        docs = [d.get('Situation', str(d)) if isinstance(d, dict) else d for d in docs_raw]

        prompt = self.prompt_builder.build(
            user_text=user_input,
            emotions=emotions,
            docs=docs,
            history=self.chat_history
        )
        response = self.llm.generate(prompt)
        self.chat_history.append({"role": "user", "content": user_input})
        self.chat_history.append({"role": "assistant", "content": response})
        self.chat_history = self.chat_history[-20:]
        return response

    def reset(self) -> None:
        self.chat_history = []
        self._emotion_history = []

    def save(self, base_path: str = "/content/drive/MyDrive/Colab Notebooks/chat") -> None:
        os.makedirs(base_path, exist_ok=True)
        # Save a dictionary containing state_dict and config for EmotionModel
        emotion_model_data = {
            "model_state_dict": self.emotion_model.state_dict(),
            "base_model": EMO_MODEL_NAME,
            "n_emo": len(self.id2emotion),
            "n_behavior": len(self.id2behavior) if self.id2behavior else None
        }
        torch.save(emotion_model_data, os.path.join(base_path, "best_emotion.pt")) # Save to best_emotion.pt
        self.emo_tokenizer.save_pretrained(os.path.join(base_path, "phobert"))
        self.retriever.save(base_path)

        with open(os.path.join(base_path, "label_maps.pkl"), "wb") as f:
            pickle.dump({"id2emotion": self.id2emotion, "id2behavior": self.id2behavior}, f)
        print(f"Saved to {base_path}/")

    @classmethod
    def from_saved(cls, base_path: str, device="cpu"):
        # 1. Load Label Maps (pkl)
        with open(os.path.join(base_path, "label_maps.pkl"), "rb") as f:
            labels = pickle.load(f)
        id2emotion = labels["id2emotion"]
        id2behavior = labels["id2behavior"]

        # 2. Load Emotion Model Checkpoint
        checkpoint_path = os.path.join(base_path, "best_emotion.pt")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Trích xuất cấu hình từ checkpoint
        base_model_name = checkpoint["base_model"]
        n_emo = checkpoint["n_emo"]
        n_behavior = checkpoint["n_behavior"]

        # 3. Khởi tạo EmotionModel (Khung xương)
        emo_model_core = EmotionModel(
            n_emo=n_emo, # Corrected argument name
            n_behavior=n_behavior,
            base_model=base_model_name # Corrected argument name
        )

        # Đổ trọng số đã lưu vào khung xương
        emo_model_core.load_state_dict(checkpoint["model_state_dict"])

        # 4. Khởi tạo các thành phần khác
        from transformers import AutoTokenizer
        emo_tokenizer = AutoTokenizer.from_pretrained(os.path.join(base_path, "phobert"))

        retriever = Retriever.load(base_path)
        llm = LLM( model_name = LLM_MODEL_NAME)

        # 5. Trả về instance của TherapyChatbot
        return cls(
            retriever=retriever,
            emotion_model=emo_model_core,
            emo_tokenizer=emo_tokenizer,
            id2emotion=id2emotion,
            id2behavior=id2behavior,
            llm=llm
        )

In [58]:
DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/chat"

def load_chatbot(
    drive_path: str = DRIVE_PATH,
) -> "TherapyChatbot":

    print(
        f"Loading chatbot từ: {drive_path}"
    )

    bot = TherapyChatbot.from_saved(
        base_path=drive_path,
    )

    print("✅ Chatbot sẵn sàng!")

    return bot

In [59]:
from transformers import AutoModel, AutoTokenizer

# Load model gốc từ HuggingFace (chưa có lớp phân loại cảm xúc của bạn)
base_tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
base_model = AutoModel.from_pretrained("vinai/phobert-base")

# Khi chạy câu "Hôm nay tôi buồn quá":
# Model này sẽ chỉ trả về các vector số học (embeddings)
# Nó hoàn toàn không biết "buồn" là nhãn số mấy trong 28 nhãn cảm xúc.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [60]:
chatbot = load_chatbot()

Loading chatbot từ: /content/drive/MyDrive/Colab Notebooks/chat


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Embedding Model: intfloat/multilingual-e5-small...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded BGE retriever: 10000 docs
Loading LLM: Qwen/Qwen2.5-3B-Instruct ...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM loaded.
✅ Chatbot sẵn sàng!


In [61]:
import gradio as gr
import sys, os
import pickle, torch

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@300;400;600;700&display=swap');
body, .gradio-container { font-family: 'Nunito', sans-serif !important; background: #f5f0ea !important; }
.therapy-header { text-align: center; padding: 20px 0; }
.comparison-label { font-weight: bold; font-size: 1.1em; margin-bottom: 10px; }
#chatbot { border-radius: 18px !important; background: #faf8f5 !important; }
"""

last_emotions = []

def get_emotion_html(emotions):
    if not emotions: return "<span style='color:#b8c0bc'>Chưa nhận diện cảm xúc</span>"
    badges = []
    for emo in emotions:
        info = EMOTION_GROUPS.get(emo.lower(), {"icon": "🌀", "color": "#94a3b8"})
        emoji, color = info["icon"], info["color"]
        badges.append(f"<span style='background:{color}22;border:1.5px solid {color}55;color:{color};border-radius:20px;padding:4px 12px;font-weight:700;margin:2px'>{emoji} {emo}</span>")
    return f"<b>Cảm xúc (chỉ model Tuned có):</b> { ' '.join(badges) }"

def respond(message, history_no_tuning, history_fine_tuned):
    global last_emotions
    if not message.strip(): return history_no_tuning, history_fine_tuned, get_emotion_html(last_emotions), ""

    # FINETUNE
    # Emotion Model + RAG + PromptBuilder
    reply_tuned = chatbot.chat(message)
    last_emotions = chatbot._emotion_history[-1] if chatbot._emotion_history else []

    # KHÔNG FINETUNE
    # gọi LLM- qwen thuần với 1 prompt cơ bản, không có phân tích cảm xúc hay RAG
    prompt_raw = f"Bạn là một trợ lý AI thông thường. Trả lời câu này: {message}"
    reply_no_tune = chatbot.llm.generate(prompt_raw)

    history_no_tuning = history_no_tuning or []
    history_no_tuning.append((message, reply_no_tune))

    history_fine_tuned = history_fine_tuned or []
    history_fine_tuned.append((message, reply_tuned))

    return history_no_tuning, history_fine_tuned, get_emotion_html(last_emotions), ""

def reset_all():
    global last_emotions
    chatbot.reset()
    last_emotions = []
    return [], [], get_emotion_html([]), ""

with gr.Blocks(css=CUSTOM_CSS, title="Chứng minh hiệu quả Fine-tune 🌿") as demo:
    gr.HTML("<div class='therapy-header'><h1>🌿 Fine-tuning Comparison</h1><p>Sự khác biệt giữa Model mặc định và Model được huấn luyện chuyên sâu</p></div>")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Phobert")
            chatbot_no_tune_ui = gr.Chatbot(label="Base Model", height=400)

        with gr.Column():
            gr.Markdown("### Fine-tune")
            chatbot_tuned_ui = gr.Chatbot(label="Tuned Model", height=400)

    with gr.Row():
        with gr.Column(scale=4):
            msg_input = gr.Textbox(placeholder="Nhập thử: 'Hôm nay mình buồn quá' hoặc 'Lại một kỳ thi nữa trôi qua'...", show_label=False)
        with gr.Column(scale=2):
            emotion_display = gr.HTML(value=get_emotion_html([]))

    with gr.Row():
        send_btn = gr.Button("So sánh ngay ✉️", variant="primary")
        reset_btn = gr.Button("Làm mới 🔄")

    # Xử lý sự kiện
    event_args = {
        "fn": respond,
        "inputs": [msg_input, chatbot_no_tune_ui, chatbot_tuned_ui],
        "outputs": [chatbot_no_tune_ui, chatbot_tuned_ui, emotion_display, msg_input]
    }
    send_btn.click(**event_args)
    msg_input.submit(**event_args)
    reset_btn.click(reset_all, None, [chatbot_no_tune_ui, chatbot_tuned_ui, emotion_display, msg_input])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4fb9a01e4ea3e3a92f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7863 <> https://4fb9a01e4ea3e3a92f.gradio.live
